In [5]:
import h5py

filename = "2025-06-03_0041_AWG_moving_0.h5"

def print_hdf5_structure(name, obj):
    print(f"{name}: {type(obj)}")

with h5py.File(filename, 'r') as f:
    f.visititems(print_hdf5_structure)
    ch0_samples = f["devices/awg/0/sample"][:]

calibrations: <class 'h5py._hl.group.Group'>
connection table: <class 'h5py._hl.dataset.Dataset'>
devices: <class 'h5py._hl.group.Group'>
devices/awg: <class 'h5py._hl.group.Group'>
devices/awg/0: <class 'h5py._hl.group.Group'>
devices/awg/0/labels: <class 'h5py._hl.group.Group'>
devices/awg/0/sample: <class 'h5py._hl.group.Group'>
devices/awg/1: <class 'h5py._hl.group.Group'>
devices/awg/1/labels: <class 'h5py._hl.group.Group'>
devices/awg/1/sample: <class 'h5py._hl.group.Group'>
devices/main_board: <class 'h5py._hl.group.Group'>
devices/main_board/main_board_matrix: <class 'h5py._hl.dataset.Dataset'>
devices/main_board/main_board_worker_args_ex: <class 'h5py._hl.dataset.Dataset'>
globals: <class 'h5py._hl.group.Group'>
globals/TweezerTest: <class 'h5py._hl.group.Group'>
globals/TweezerTest/expansion: <class 'h5py._hl.group.Group'>
globals/TweezerTest/units: <class 'h5py._hl.group.Group'>
labscriptlib: <class 'h5py._hl.group.Group'>
labscriptlib/Test_AWG_Spectrum: <class 'h5py._hl.gro

TypeError: Accessing a group is done with bytes or str, not <class 'slice'>

In [6]:

import h5py

filename = "2025-06-03_0041_AWG_moving_0.h5"

try:
    with h5py.File(filename, 'r') as f:
        print("File opened successfully.")
        for key in f.keys():
            print(f"Top-level key: {key}")
except Exception as e:
    print(f"Failed to open HDF5 file: {e}")

File opened successfully.
Top-level key: calibrations
Top-level key: connection table
Top-level key: devices
Top-level key: globals
Top-level key: labscriptlib
Top-level key: script
Top-level key: shot_properties
Top-level key: time_markers
Top-level key: waits


In [7]:
import h5py

filename = "2025-06-03_0041_AWG_moving_0.h5"

with h5py.File(filename, 'r') as f:
    print("Contents of 'devices':")
    for dev in f['devices'].keys():
        print(f"  - Device: {dev}")
        if isinstance(f['devices'][dev], h5py.Group):
            for ch in f['devices'][dev].keys():
                print(f"    - {ch}")
                if isinstance(f['devices'][dev][ch], h5py.Group):
                    for subkey in f['devices'][dev][ch].keys():
                        print(f"      - {subkey}")

Contents of 'devices':
  - Device: awg
    - 0
      - labels
      - sample
    - 1
      - labels
      - sample
  - Device: main_board
    - main_board_matrix
    - main_board_worker_args_ex


In [8]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft

# === Settings ===
filename = "2025-06-03_0041_AWG_moving_0.h5"
sample_rate = 1.25e9  # Hz, adjust to your AWG's actual sampling rate

# === Load HDF5 file and extract samples ===
with h5py.File(filename, 'r') as f:
    ch0_samples = f['devices/awg/0/sample'][:]
    ch1_samples = f['devices/awg/1/sample'][:]

# === Normalize samples (assume int16) ===
ch0 = ch0_samples.astype(np.float32) / np.iinfo(ch0_samples.dtype).max
ch1 = ch1_samples.astype(np.float32) / np.iinfo(ch1_samples.dtype).max

# === Short-Time Fourier Transform (STFT) ===
nperseg = 4096
noverlap = nperseg // 2

f0, t0, Zxx0 = stft(ch0, fs=sample_rate, nperseg=nperseg, noverlap=noverlap)
f1, t1, Zxx1 = stft(ch1, fs=sample_rate, nperseg=nperseg, noverlap=noverlap)

# === Plotting ===
plt.figure(figsize=(12, 8))

# Channel 0 Spectrogram
plt.subplot(2, 1, 1)
plt.pcolormesh(t0 * 1e6, f0 / 1e6, np.abs(Zxx0), shading='gouraud')
plt.title("Channel 0 – Spectrogram")
plt.ylabel("Frequency (MHz)")
plt.xlabel("Time (µs)")
plt.colorbar(label='Amplitude')

# Channel 1 Spectrogram
plt.subplot(2, 1, 2)
plt.pcolormesh(t1 * 1e6, f1 / 1e6, np.abs(Zxx1), shading='gouraud')
plt.title("Channel 1 – Spectrogram")
plt.ylabel("Frequency (MHz)")
plt.xlabel("Time (µs)")
plt.colorbar(label='Amplitude')

plt.tight_layout()
plt.show()


TypeError: Accessing a group is done with bytes or str, not <class 'slice'>

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft

# === Settings ===
filename = "2025-06-03_0041_AWG_moving_0.h5"
sample_rate = 1.25e9  # Hz

# === Load HDF5 file and extract samples ===
with h5py.File(filename, 'r') as f:
    # Use full string keys here (no slicing!)
    ch0_samples = f['devices']['awg']['0']['sample'][()]
    ch1_samples = f['devices']['awg']['1']['sample'][()]

# === Normalize samples ===
ch0 = ch0_samples.astype(np.float32) / np.iinfo(ch0_samples.dtype).max
ch1 = ch1_samples.astype(np.float32) / np.iinfo(ch1_samples.dtype).max

# === Short-Time Fourier Transform (STFT) ===
nperseg = 4096
noverlap = nperseg // 2

f0, t0, Zxx0 = stft(ch0, fs=sample_rate, nperseg=nperseg, noverlap=noverlap)
f1, t1, Zxx1 = stft(ch1, fs=sample_rate, nperseg=nperseg, noverlap=noverlap)

# === Plotting ===
plt.figure(figsize=(12, 8))

# Channel 0 Spectrogram
plt.subplot(2, 1, 1)
plt.pcolormesh(t0 * 1e6, f0 / 1e6, np.abs(Zxx0), shading='gouraud')
plt.title("Channel 0 – Spectrogram")
plt.ylabel("Frequency (MHz)")
plt.xlabel("Time (µs)")
plt.colorbar(label='Amplitude')

# Channel 1 Spectrogram
plt.subplot(2, 1, 2)
plt.pcolormesh(t1 * 1e6, f1 / 1e6, np.abs(Zxx1), shading='gouraud')
plt.title("Channel 1 – Spectrogram")
plt.ylabel("Frequency (MHz)")
plt.xlabel("Time (µs)")
plt.colorbar(label='Amplitude')

plt.tight_layout()
plt.show()
